# Copying Congress Trades — port CoursIA

Port du research note QuantConnect [17886](https://www.quantconnect.com/research/17886/copying-congress-trades/) (Derek Melchin). Grain #16372, Epic #11698.

Ce notebook documente le port : mecanisme, parametrage, chaine diagnostique QC Cloud et verdict SOTA honnete. Projet QC Cloud : `Congress-Trades-Copy` (id 36717556), compile `BuildSuccess`.

## Mecanisme

- **Univers** : actions recemment achetees par des membres du Congres US (dataset [Quiver Quantitative US Congress Trading](https://www.quantconnect.com/docs/v2/writing-algorithms/datasets/quiver-quantitative/us-congress-trading), payant ; disclosures STOCK Act <= 45 jours ; coverage 2016+, 1800 equities, frequence quotidienne). Seules les transactions BUY declenchent l'inclusion.
- **Rebalancement** : hebdomadaire, premier jour ouvre, 30 min apres l'open SPY.
- **Construction** : pondervation inverse-volatilite (vol quotidienne trailing ~6 mois), levier cible 1,5x, cap 10 % par actif.
- **Execution** : `set_holdings(targets, True)`.

**Fricition corrigee** : `self._universe.selected` retourne `None` en certains contextes (rapporte par L. Raducu en live, commentaires de l'article) -- le port cache la selection dans le selecteur (`self._selected`).

## Plan de mesure prevu

L'article divulgue Sharpe algo 0,934 vs SPY 0,7 mais **pas** la periode, le drawdown, le turnover, ni la robustesse. Jambe prevue :

| Run | Periode | levier | cap | vol-window | Role |
|---|---|---|---|---|---|
| baseline | 2019-2024 | 1,5 | 0,10 | 180 | reproduction |
| oos | 2025-present | 1,5 | 0,10 | 180 | jambe OOS distincte |
| lev1 / lev2 | 2019-2024 | 1,0 / 2,0 | 0,10 | 180 | sensibilite levier |
| cap05 / cap20 | 2019-2024 | 1,5 | 0,05 / 0,20 | 180 | sensibilite cap |
| vol90 / vol360 | 2019-2024 | 1,5 | 0,10 | 90 / 360 | sensibilite fenetre |

Tous les parametres (`leverage`, `cap`, `vol-window`, `start`, `end`) sont exposes au backtest QC -- la serie complete se lance sans recompilation une fois le dataset accessible.

## Chaine diagnostique : pourquoi les backtests ne tradent pas

La premiere baseline (2019-2024) complete avec **0 date tradable et 0 ordre**. Quatre sondes croisees sur QC Cloud (logs non lisibles via l'API utilisee -- diagnostics encodes dans les observables disponibles) :

| Probe | Discriminateur | Resultat |
|---|---|---|
| run1 baseline | erreur runtime en fin de run | erreur au log final (`transactions_count` inexistant -> corrige en `len(transaction_record)`), mais surtout 0 tradeable dates | 
| run2 baseline corrigee | run complet | **0 tradeable date, 0 ordre** sur 6 ans |
| probe2 Q1-2019 | SPY souscrit explicitement (61 dates tradables prouvees) + vraie logique de trading | **0 ordre** : `self._selected` toujours vide |
| probe3 Q1-2019 | ordre hebdo inconditionnel (encodage par quantite) | **12 ordres** : le schedule hebdo fonctionne |
| probe4 Q1-2019 | ordre place **ssi** le callback de selection d'univers a tire | **0 ordre** : le selecteur n'est **jamais** appele |

In [1]:
# Faits mesures (chaque ligne = observable QC Cloud, ids captures)
facts = [
    {"probe": "run2", "observable": "0 tradeable dates / 0 orders sur 2019-2024", "id": "13909a0110583b4ebbf7a9d11f6f3502"},
    {"probe": "probe2", "observable": "61 tradeable dates (SPY souscrit) / 0 orders", "id": "4c5719946c525f6ea4ff5b1be98894d4"},
    {"probe": "probe3", "observable": "12 orders -> schedule hebdo OK", "id": "eaaf4105a658e21a60a0926500722cd7"},
    {"probe": "probe4", "observable": "0 orders -> callback de selection jamais appele", "id": "82e180c544ee6615d40e84e8e19452ce"},
]
facts


[{'probe': 'run2',
  'observable': '0 tradeable dates / 0 orders sur 2019-2024',
  'id': '13909a0110583b4ebbf7a9d11f6f3502'},
 {'probe': 'probe2',
  'observable': '61 tradeable dates (SPY souscrit) / 0 orders',
  'id': '4c5719946c525f6ea4ff5b1be98894d4'},
 {'probe': 'probe3',
  'observable': '12 orders -> schedule hebdo OK',
  'id': 'eaaf4105a658e21a60a0926500722cd7'},
 {'probe': 'probe4',
  'observable': '0 orders -> callback de selection jamais appele',
  'id': '82e180c544ee6615d40e84e8e19452ce'}]

### Interpretation

Le code est **doc-exact** (l'exemple officiel du dataset utilise la meme signature `add_universe(QuiverQuantCongressUniverse, selector)` et le meme idiome `d.transaction == OrderDirection.BUY`), la **coverage documentee commence en janvier 2016** (Q1-2019 couvert), et la compilation est verte. Un univers qui ne tire **jamais** son callback sur une periode couverte, avec un code doc-exact, est la signature d'un **dataset non entitle** pour l'organisation : le feed ne livre rien, silencieusement (pas d'erreur d'acces en backtest -- comportement mesure, conforme au modele de datasets payants QC actives par organisation).

## Verdict SOTA : RECOVERABLE-USER-HAND

Un seul geste debloque la jambe backtest complete : **souscrire le dataset *US Congress Trading* (Quiver Quantitative) pour l'organisation QC** (`d600793e…`, Data Library -> Quiver Quantitative -> US Congress Trading). Des l'entitlement actif, la serie prevue ci-dessus (baseline, OOS, 6 sensibilites) se lance tel quel -- les parametres sont exposes.

Ni workaround (le port n'embarque aucune donnee fabriquee), ni INTRINSIC (le dataset existe et est invocable). Inscription au registre des questions user : demande de souscription, verifiable par la reprise du meme backtest baseline qui doit alors produire des ordres.

## Contre-evidence et limites (lecture 5-axes du grain)

- Thian Seong Yee (commentaire de l'article) : rendements similaires au buy-and-hold SPY, drawdown plus faible, Sharpe meilleur -- coherent avec un overlay de risque plus qu'avec un alpha fort.
- L. Raducu (commentaire) : `selected = None` en live + exception RabbitMQ `RESOURCE_LOCKED` -- frictions d'implementation reelles sur le path live.
- L'article ne cite **aucun papier academique** (gap documente dans le grain) ; la litterature sur le STOCK Act (Eggers & Hainmueller 2014 ; Ziobrowski et al.) conteste l'edge net des trades du Congres.
- Biais de publication : l'auteur est QuantConnect (promotion du dataset payant).
- Le verdict BEATS/NO BEATS reste **INCONCLUSIF tant que l'entitlement n'est pas actif** -- aucune metrique n'a ete mesuree.